<div style="background:linear-gradient(135deg,#6C63FF 0%,#00B8A9 100%);padding:34px 30px;border-radius:14px;color:white;">
<h1 style="margin:0;color:white;">🎓 Student Performance Analytics</h1>
<h3 style="margin:8px 0 0 0;color:#f0f0ff;font-weight:400;">What really drives final grades in Mathematics and Portuguese?</h3>
<p style="margin:14px 0 0 0;color:#e8e8ff;">Data Analytics Final Project &nbsp;|&nbsp; End-to-end Exploratory Data Analysis in Python</p>
</div>

| | |
|---|---|
| **Domain** | Education – Student performance dashboard |
| **Dataset** | Student Performance data (secondary-school students, Portugal) – Mathematics + Portuguese combined |
| **Records / Features** | 1,044 rows × 34 columns (mix of numerical and categorical) |
| **Tools** | Python · Pandas · NumPy |

---

<h2 style="color:#5B52E0;border-bottom:3px solid #5B52E0;padding-bottom:6px;margin-top:30px;">📑 Table of Contents</h2>
0. [Introduction to the Project](#0)
1. [Data Loading & Initial Overview](#1)
2. [Data Cleaning & Pre-processing](#2)
3. [Data Exploration (tables & summary)](#3)

<a id="0"></a>
<h1 style="background:linear-gradient(90deg,#6C63FF,#3A86FF);color:white;padding:16px 24px;border-radius:12px;margin:34px 0 12px 0;box-shadow:0 3px 8px rgba(0,0,0,.18);">0️⃣ Introduction to the Project</h1>

<h2 style="color:#5B52E0;border-bottom:3px solid #5B52E0;padding-bottom:6px;margin-top:30px;">🌍 Background</h2>
Every school year, teachers see some students move ahead easily while others quietly fall behind. By the time the **final grade** is published, it is too late to help. Schools collect a lot of information about their students (grades, attendance, study habits, family background, free-time activities), but it is rarely analysed together.

This project uses **data analytics in Python** to find out *what is really connected to a student's final result*, so that a school can notice problems **early** and decide where support will matter most.

<h2 style="color:#5B52E0;border-bottom:3px solid #5B52E0;padding-bottom:6px;margin-top:30px;">🎯 Problem Statement</h2>
> **Which student, family, school and lifestyle factors are associated with a higher or lower final grade (`G3`) in Mathematics and Portuguese, and which of them can a school realistically act on?**

<h2 style="color:#5B52E0;border-bottom:3px solid #5B52E0;padding-bottom:6px;margin-top:30px;">✅ Objectives</h2>
1. **Clean** the data so that the analysis is reliable (missing values, duplicates, data types, outliers).
2. **Explore** the data with summary tables, group comparisons, pivot tables and correlations, and summarise the patterns found.

<h2 style="color:#5B52E0;border-bottom:3px solid #5B52E0;padding-bottom:6px;margin-top:30px;">📦 About the Dataset</h2>
The file combines the records of students from **two Portuguese secondary schools** (Gabriel Pereira – `GP`, Mousinho da Silveira – `MS`), with one row per *student–subject* enrolment (Mathematics or Portuguese language). It contains demographic, social and school-related attributes collected through school reports and questionnaires.

| Group | Columns (original name → meaning) |
|---|---|
| **Identity / context** | `subject`, `school`, `sex`, `age`, `address` (urban/rural), `famsize`, `Pstatus` (parents together/apart) |
| **Family background** | `Medu`, `Fedu` (parents' education 0–4), `Mjob`, `Fjob`, `guardian`, `famrel` (family relationship quality 1–5) |
| **School & support** | `reason` (why this school), `traveltime` (1–4), `studytime` (1–4), `failures` (past class failures), `schoolsup`, `famsup`, `paid` (extra paid classes), `activities`, `nursery`, `higher` (wants higher education), `internet` |
| **Lifestyle** | `romantic`, `freetime`, `goout`, `Dalc` (weekday alcohol 1–5), `Walc` (weekend alcohol 1–5), `health` (1–5) |
| **Performance** | `absences`, `G1`, `G2` (period grades), **`G3` (final grade, 0–20 – target variable)** |

**Meets the project requirements:** 1,044 records (≥ 500) and 34 columns (≥ 10), with both numerical and categorical variables.

> ⚠️ **Note on the data:** because there is no student ID, a student who takes both subjects appears twice. Rows are therefore *student–subject enrolments*, not unique students. This is kept in mind when interpreting results.

<h2 style="color:#5B52E0;border-bottom:3px solid #5B52E0;padding-bottom:6px;margin-top:30px;">🗺️ Project Roadmap</h2>
The notebook is organised in **separate parts**, each with a different purpose:

| Part | Name | What happens here |
|---|---|---|
| **1** | Data Loading | Import the CSV, look at its size, data types and first rows |
| **2** | Data Cleaning | Fix the data: missing values, duplicates, types, new columns, outliers |
| **3** | Data Exploration | *Numbers only*: summary tables, `groupby`, pivot tables, correlations, and a summary of the patterns found |

<h2 style="color:#5B52E0;border-bottom:3px solid #5B52E0;padding-bottom:6px;margin-top:30px;">🧭 How to Read this Notebook</h2>
* A short **Markdown explanation** comes before each step, and an **interpretation** comes after each result.
* Part 3 follows the order **one variable → two variables → many variables**.

<h2 style="color:#5B52E0;border-bottom:3px solid #5B52E0;padding-bottom:6px;margin-top:30px;">⚙️ Environment Setup</h2>
Importing libraries and defining a consistent visual theme used across the whole notebook.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from IPython.display import display, Markdown

# ---- Visual theme -------------------------------------------------------
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.05)
COLORS = {"Math": "#6C63FF", "Portuguese": "#00B8A9"}          # subject colours
ACCENT = ["#6C63FF", "#00B8A9", "#F8615A", "#FFB400", "#3A86FF", "#8338EC"]
sns.set_palette(ACCENT)

plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)

def annotate_bars(ax, fmt="{:.0f}", fontsize=10):
    """Write the value on top of every bar of an axis."""
    for p in ax.patches:
        h = p.get_height()
        if np.isfinite(h) and h > 0:
            ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2, h),
                        ha="center", va="bottom", fontsize=fontsize, fontweight="bold")

print("Libraries loaded ✔")

Libraries loaded ✔


<a id="1"></a>
<h1 style="background:linear-gradient(90deg,#3A86FF,#00B8A9);color:white;padding:16px 24px;border-radius:12px;margin:34px 0 12px 0;box-shadow:0 3px 8px rgba(0,0,0,.18);">1️⃣ Data Loading & Initial Overview</h1>
We import the raw CSV with Pandas and inspect its size, structure, data types and basic statistics **before changing anything**.

In [2]:
# If you run this in Google Colab, upload the file first:
#   from google.colab import files; files.upload()
DATA_PATH = "student_performance.csv"

raw = pd.read_csv(DATA_PATH)          # untouched copy of the original data
df = raw.copy()                       # working copy used for cleaning

print(f"📊 Rows    : {df.shape[0]:,}")
print(f"📊 Columns : {df.shape[1]}")

📊 Rows    : 1,044
📊 Columns : 34


<h3 style="color:#0E9F95;border-left:6px solid #0E9F95;padding-left:12px;margin-top:22px;">1.1 First look – `head()`</h3>

In [3]:
df.head(10)

,subject,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,reason,guardian,traveltime,studytime,failures,schoolsup,famsup,paid,activities,nursery,higher,internet,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3
0,Math,GP,F,18,U,GT3,A,4,4,at_home,teacher,course,mother,2,2,0,yes,no,no,no,yes,yes,no,no,4,3,4,1,1,3,6,5,6,6
1,Math,GP,F,17,U,GT3,T,1,1,at_home,other,course,father,1,2,0,no,yes,no,no,no,yes,yes,no,5,3,3,1,1,3,4,5,5,6
2,Math,GP,F,15,U,LE3,T,1,1,at_home,other,other,mother,1,2,3,yes,no,yes,no,yes,yes,yes,no,4,3,2,2,3,3,10,7,8,10
3,Math,GP,F,15,U,GT3,T,4,2,health,services,home,mother,1,3,0,no,yes,yes,yes,yes,yes,yes,yes,3,2,2,1,1,5,2,15,14,15
4,Math,GP,F,16,U,GT3,T,3,3,other,other,home,father,1,2,0,no,yes,yes,no,yes,yes,no,no,4,3,2,1,2,5,4,6,10,10
5,Math,GP,M,16,U,LE3,T,4,3,services,other,reputation,mother,1,2,0,no,yes,yes,yes,yes,yes,yes,no,5,4,2,1,2,5,10,15,15,15
6,Math,GP,M,16,U,LE3,T,2,2,other,other,home,mother,1,2,0,no,no,no,no,yes,yes,yes,no,4,4,4,1,1,3,0,12,12,11
7,Math,GP,F,17,U,GT3,A,4,4,other,teacher,home,mother,2,2,0,yes,yes,no,no,yes,yes,no,no,4,1,4,1,1,1,6,6,5,6
8,Math,GP,M,15,U,LE3,A,3,2,services,other,home,mother,1,2,0,no,yes,yes,no,yes,yes,yes,no,4,2,2,1,1,1,0,16,18,19
9,Math,GP,M,15,U,GT3,T,3,4,other,other,home,mother,1,2,0,no,yes,yes,yes,yes,yes,yes,no,5,5,1,1,1,5,0,14,15,15


<h3 style="color:#0E9F95;border-left:6px solid #0E9F95;padding-left:12px;margin-top:22px;">1.2 Structure and data types – `info()`</h3>

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1044 entries, 0 to 1043
Data columns (total 34 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   subject     1044 non-null   str  
 1   school      1044 non-null   str  
 2   sex         1044 non-null   str  
 3   age         1044 non-null   int64
 4   address     1044 non-null   str  
 5   famsize     1044 non-null   str  
 6   Pstatus     1044 non-null   str  
 7   Medu        1044 non-null   int64
 8   Fedu        1044 non-null   int64
 9   Mjob        1044 non-null   str  
 10  Fjob        1044 non-null   str  
 11  reason      1044 non-null   str  
 12  guardian    1044 non-null   str  
 13  traveltime  1044 non-null   int64
 14  studytime   1044 non-null   int64
 15  failures    1044 non-null   int64
 16  schoolsup   1044 non-null   str  
 17  famsup      1044 non-null   str  
 18  paid        1044 non-null   str  
 19  activities  1044 non-null   str  
 20  nursery     1044 non-null   str  
 21  hi

<h3 style="color:#0E9F95;border-left:6px solid #0E9F95;padding-left:12px;margin-top:22px;">1.3 Data types at a glance</h3>

In [5]:
dtype_summary = (
    pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "unique_values": df.nunique(),
        "example": df.iloc[0],
    })
    .rename_axis("column")
)
dtype_summary

,dtype,non_null,unique_values,example
column,,,,
subject,str,1044,2,Math
school,str,1044,2,GP
sex,str,1044,2,F
age,int64,1044,8,18
address,str,1044,2,U
famsize,str,1044,2,GT3
Pstatus,str,1044,2,A
Medu,int64,1044,5,4
Fedu,int64,1044,5,4


<h3 style="color:#0E9F95;border-left:6px solid #0E9F95;padding-left:12px;margin-top:22px;">1.4 Statistical summary – `describe()`</h3>

In [6]:
print("NUMERICAL COLUMNS")
display(df.describe().T)
print("\nCATEGORICAL COLUMNS")
display(df.describe(exclude="number").T)

NUMERICAL COLUMNS


,count,mean,std,min,25%,50%,75%,max
age,1044.00,16.73,1.24,15.00,16.00,17.00,18.00,22.00
Medu,1044.00,2.60,1.12,0.00,2.00,3.00,4.00,4.00
Fedu,1044.00,2.39,1.10,0.00,1.00,2.00,3.00,4.00
traveltime,1044.00,1.52,0.73,1.00,1.00,1.00,2.00,4.00
studytime,1044.00,1.97,0.83,1.00,1.00,2.00,2.00,4.00
failures,1044.00,0.26,0.66,0.00,0.00,0.00,0.00,3.00
famrel,1044.00,3.94,0.93,1.00,4.00,4.00,5.00,5.00
freetime,1044.00,3.20,1.03,1.00,3.00,3.00,4.00,5.00
goout,1044.00,3.16,1.15,1.00,2.00,3.00,4.00,5.00
Dalc,1044.00,1.49,0.91,1.00,1.00,1.00,2.00,5.00



CATEGORICAL COLUMNS


,count,unique,top,freq
subject,1044,2,Portuguese,649
school,1044,2,GP,772
sex,1044,2,F,591
address,1044,2,U,759
famsize,1044,2,GT3,738
Pstatus,1044,2,T,923
Mjob,1044,5,other,399
Fjob,1044,5,other,584
reason,1044,4,course,430
guardian,1044,3,mother,728


<h3 style="color:#0E9F95;border-left:6px solid #0E9F95;padding-left:12px;margin-top:22px;">🔎 Initial Observations</h3>
* The dataset has **1,044 rows and 34 columns**: 16 numeric (integers) and 18 categorical (text) columns, so it meets the size and variety requirements.
* **No missing values** appear in `info()`, and there are no obvious duplicates (verified formally in the next section).
* Grades `G1`, `G2`, `G3` range from 0 to 20. `G3` has a minimum of **0**, which deserves a closer look (a grade of 0 is unusual compared with the rest of the distribution).
* `absences` has a maximum of **75** while the median is only 2, so the column is strongly right-skewed and likely contains outliers.
* Most "rating" variables (`famrel`, `freetime`, `goout`, `Dalc`, `Walc`, `health`, `studytime`, `traveltime`, `Medu`, `Fedu`) are **ordinal scales** stored as integers. They are numeric in type but categorical in meaning.
* Column names such as `Medu`, `Dalc` or `famsup` are cryptic. Renaming them will make the rest of the notebook easier to read.

<a id="2"></a>
<h1 style="background:linear-gradient(90deg,#00B8A9,#22C55E);color:white;padding:16px 24px;border-radius:12px;margin:34px 0 12px 0;box-shadow:0 3px 8px rgba(0,0,0,.18);">2️⃣ Data Cleaning & Pre-processing</h1>
This part only **prepares** the data; no conclusions are drawn yet. Cleaning plan:

| Step | What we do |
|---|---|
| 2.1 | Check and handle **missing values** and **duplicates**; validate value ranges |
| 2.2 | Make the data readable: **rename** columns and **decode** labels |
| 2.3 | **Correct data types** (categories, compact integers) |
| 2.4 | Create **derived columns** (feature engineering) |
| 2.5 | Detect and decide how to treat **outliers** |
| 2.6 | **Filter / aggregate** the data for later analysis |

<h3 style="color:#16A34A;border-left:6px solid #16A34A;padding-left:12px;margin-top:22px;">2.1 Missing values, duplicates and validity checks</h3>

In [7]:
# --- Missing values ---------------------------------------------------------
missing = df.isna().sum()
print(f"Total missing values : {missing.sum()}")

# --- Duplicate rows ----------------------------------------------------------
n_dups = df.duplicated().sum()
print(f"Duplicate rows       : {n_dups}")
df = df.drop_duplicates().reset_index(drop=True)

# --- Whitespace / inconsistent text -------------------------------------------
text_cols = df.select_dtypes(exclude="number").columns
for c in text_cols:
    df[c] = df[c].str.strip()

# --- Range validation (data-quality rules from the data dictionary) ----------
rules = {
    "Grades G1-G3 within 0-20":        df[["G1", "G2", "G3"]].apply(lambda s: s.between(0, 20)).all().all(),
    "Parent education within 0-4":     df[["Medu", "Fedu"]].apply(lambda s: s.between(0, 4)).all().all(),
    "Rating scales within 1-5":        df[["famrel", "freetime", "goout", "Dalc", "Walc", "health"]].apply(lambda s: s.between(1, 5)).all().all(),
    "studytime / traveltime in 1-4":   df[["studytime", "traveltime"]].apply(lambda s: s.between(1, 4)).all().all(),
    "failures within 0-3":             df["failures"].between(0, 3).all(),
    "absences are non-negative":       (df["absences"] >= 0).all(),
}
pd.DataFrame({"Rule passed?": rules}).replace({True: "✅ Yes", False: "❌ No"})

Total missing values : 0
Duplicate rows       : 0


,Rule passed?
Grades G1-G3 within 0-20,✅ Yes
Parent education within 0-4,✅ Yes
Rating scales within 1-5,✅ Yes
studytime / traveltime in 1-4,✅ Yes
failures within 0-3,✅ Yes
absences are non-negative,✅ Yes


**Decision:** the data contains **no missing values, no duplicate rows and no out-of-range values**, so no rows need to be imputed or removed at this stage. We only trimmed whitespace in text columns as a safeguard.

<h3 style="color:#16A34A;border-left:6px solid #16A34A;padding-left:12px;margin-top:22px;">2.2 Renaming columns and decoding labels</h3>

In [8]:
rename_map = {
    "sex": "gender", "famsize": "family_size", "Pstatus": "parents_status",
    "Medu": "mother_edu", "Fedu": "father_edu", "Mjob": "mother_job", "Fjob": "father_job",
    "reason": "school_reason", "traveltime": "travel_time", "studytime": "study_time",
    "schoolsup": "school_support", "famsup": "family_support", "paid": "paid_classes",
    "activities": "extracurricular", "nursery": "attended_nursery", "higher": "wants_higher_edu",
    "internet": "internet_access", "romantic": "in_relationship", "famrel": "family_relation",
    "freetime": "free_time", "goout": "go_out", "Dalc": "weekday_alcohol", "Walc": "weekend_alcohol",
}
df = df.rename(columns=rename_map)

# Decode abbreviations into readable labels
decode = {
    "school":         {"GP": "Gabriel Pereira", "MS": "Mousinho da Silveira"},
    "gender":         {"F": "Female", "M": "Male"},
    "address":        {"U": "Urban", "R": "Rural"},
    "family_size":    {"GT3": "Greater than 3", "LE3": "3 or less"},
    "parents_status": {"T": "Together", "A": "Apart"},
}
for col, mapping in decode.items():
    df[col] = df[col].map(mapping)

# yes/no columns -> Yes/No
yes_no_cols = ["school_support", "family_support", "paid_classes", "extracurricular",
               "attended_nursery", "wants_higher_edu", "internet_access", "in_relationship"]
for col in yes_no_cols:
    df[col] = df[col].str.title()

# nominal text columns -> tidy capitalisation
for col in ["mother_job", "father_job", "school_reason", "guardian"]:
    df[col] = df[col].str.replace("_", " ").str.title()

df.head(3)

,subject,school,gender,age,address,family_size,parents_status,mother_edu,father_edu,mother_job,father_job,school_reason,guardian,travel_time,study_time,failures,school_support,family_support,paid_classes,extracurricular,attended_nursery,wants_higher_edu,internet_access,in_relationship,family_relation,free_time,go_out,weekday_alcohol,weekend_alcohol,health,absences,G1,G2,G3
0,Math,Gabriel Pereira,Female,18,Urban,Greater than 3,Apart,4,4,At Home,Teacher,Course,Mother,2,2,0,Yes,No,No,No,Yes,Yes,No,No,4,3,4,1,1,3,6,5,6,6
1,Math,Gabriel Pereira,Female,17,Urban,Greater than 3,Together,1,1,At Home,Other,Course,Father,1,2,0,No,Yes,No,No,No,Yes,Yes,No,5,3,3,1,1,3,4,5,5,6
2,Math,Gabriel Pereira,Female,15,Urban,3 or less,Together,1,1,At Home,Other,Other,Mother,1,2,3,Yes,No,Yes,No,Yes,Yes,Yes,No,4,3,2,2,3,3,10,7,8,10


<h3 style="color:#16A34A;border-left:6px solid #16A34A;padding-left:12px;margin-top:22px;">2.3 Correcting data types</h3>
* Text columns with a small set of repeating values are converted to the memory-efficient **`category`** dtype.
* Ordinal scales are kept as integers on purpose: this lets us compute correlations and averages on them.
* Integer columns are down-cast to the smallest suitable size.

In [9]:
nominal_cols = ["subject", "school", "gender", "address", "family_size", "parents_status",
                "mother_job", "father_job", "school_reason", "guardian"] + yes_no_cols
for col in nominal_cols:
    df[col] = df[col].astype("category")

int_cols = df.select_dtypes(include="integer").columns
df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast="integer")

before = raw.memory_usage(deep=True).sum() / 1024
after = df.memory_usage(deep=True).sum() / 1024
print(f"Memory usage: {before:,.0f} KB  →  {after:,.0f} KB  ({1 - after / before:.0%} smaller)")
print()
print(df.dtypes.value_counts())

Memory usage: 1,092 KB  →  37 KB  (97% smaller)



int8        16
category     8
category     2
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


<h3 style="color:#16A34A;border-left:6px solid #16A34A;padding-left:12px;margin-top:22px;">2.4 Creating derived columns (feature engineering)</h3>
New variables make patterns easier to see:

| New column | Definition | Why it is useful |
|---|---|---|
| `avg_grade` | mean of G1, G2, G3 | overall level across the year |
| `grade_change` | `G3 − G1` | progress during the year |
| `trend` | Declined (≤ −2) / Stable (−1…+1) / Improved (≥ +2) | simple categorical view of progress |
| `pass_fail` | **Pass** if `G3 ≥ 10`, otherwise **Fail** | 10/20 is the passing mark in the Portuguese system |
| `performance_band` | Fail (0–9), Average (10–13), Good (14–16), Excellent (17–20) | grouped view of the target |
| `total_alcohol` | weighted weekly alcohol use = (5·weekday + 2·weekend) / 7 | single lifestyle indicator |
| `parents_edu_avg` | mean of mother's and father's education | household education level |
| `support_score` | number of support types received (school, family, paid classes) | amount of academic support |
| `absence_group` | None / 1–5 / 6–10 / 11–20 / 20+ | grouped absences |
| `age_group` | 15–16 / 17–18 / 19+ | age bands |
| `zero_grade` | `True` if `G3 == 0` | flags suspicious final grades |

In [10]:
df["avg_grade"]    = df[["G1", "G2", "G3"]].mean(axis=1).round(2)
df["grade_change"] = df["G3"] - df["G1"]
df["trend"] = pd.cut(df["grade_change"], bins=[-np.inf, -2, 1, np.inf],
                     labels=["Declined", "Stable", "Improved"])

df["pass_fail"] = np.where(df["G3"] >= 10, "Pass", "Fail")
df["performance_band"] = pd.cut(df["G3"], bins=[-1, 9, 13, 16, 20],
                                labels=["Fail (0-9)", "Average (10-13)", "Good (14-16)", "Excellent (17-20)"])

df["total_alcohol"]   = ((df["weekday_alcohol"] * 5 + df["weekend_alcohol"] * 2) / 7).round(2)
df["parents_edu_avg"] = (df["mother_edu"] + df["father_edu"]) / 2
df["support_score"]   = (df[["school_support", "family_support", "paid_classes"]] == "Yes").sum(axis=1)

df["absence_group"] = pd.cut(df["absences"], bins=[-1, 0, 5, 10, 20, np.inf],
                             labels=["None", "1-5", "6-10", "11-20", "20+"])
df["age_group"] = pd.cut(df["age"], bins=[14, 16, 18, 22], labels=["15-16", "17-18", "19+"])
df["zero_grade"] = df["G3"].eq(0)

df[["subject", "G1", "G2", "G3", "avg_grade", "grade_change", "trend", "pass_fail",
    "performance_band", "total_alcohol", "support_score", "absence_group"]].head(8)

,subject,G1,G2,G3,avg_grade,grade_change,trend,pass_fail,performance_band,total_alcohol,support_score,absence_group
0,Math,5,6,6,5.67,1,Stable,Fail,Fail (0-9),1.00,1,6-10
1,Math,5,5,6,5.33,1,Stable,Fail,Fail (0-9),1.00,1,1-5
2,Math,7,8,10,8.33,3,Improved,Pass,Average (10-13),2.29,2,6-10
3,Math,15,14,15,14.67,0,Stable,Pass,Good (14-16),1.00,2,1-5
4,Math,6,10,10,8.67,4,Improved,Pass,Average (10-13),1.29,2,1-5
5,Math,15,15,15,15.00,0,Stable,Pass,Good (14-16),1.29,2,6-10
6,Math,12,12,11,11.67,-1,Stable,Pass,Average (10-13),1.00,0,None
7,Math,6,5,6,5.67,0,Stable,Fail,Fail (0-9),1.00,2,6-10


<h3 style="color:#16A34A;border-left:6px solid #16A34A;padding-left:12px;margin-top:22px;">2.5 Outlier detection and treatment (IQR method)</h3>

In [11]:
def iqr_report(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((s < lo) | (s > hi)).sum()
    return pd.Series({"lower_fence": lo, "upper_fence": hi, "outliers": n_out,
                      "share_%": n_out / len(s) * 100, "max": s.max()})

outliers = pd.DataFrame({c: iqr_report(df[c]) for c in ["absences", "age", "G1", "G2", "G3", "failures"]}).T
display(outliers)

# Absences: the 99th percentile
p99 = df["absences"].quantile(0.99)
print(f"\n99th percentile of absences = {p99:.1f}  |  maximum = {df['absences'].max()}")

,lower_fence,upper_fence,outliers,share_%,max
absences,-9.00,15.00,54.00,5.17,75.00
age,13.00,21.00,2.00,0.19,22.00
G1,3.00,19.00,1.00,0.10,19.00
G2,3.00,19.00,20.00,1.92,19.00
G3,4.00,20.00,54.00,5.17,20.00
failures,0.00,0.00,183.00,17.53,3.00



99th percentile of absences = 25.6  |  maximum = 75


**Decision on outliers**

* `absences` has the most outliers (a few students missed 40–75 classes). These values are *genuine* (long illness or dropping out), so we **keep** them and instead create a **winsorised copy** (`absences_capped`, capped at the 99th percentile) for plots and correlations that are sensitive to extreme values.
* Ages above 19 and grades of 0 are rare but valid. Grade `0` is investigated in detail in the EDA because it behaves differently from the rest of the grades.

In [12]:
df["absences_capped"] = df["absences"].clip(upper=p99)
print("Original absences  → mean:", round(df["absences"].mean(), 2), "| std:", round(df["absences"].std(), 2))
print("Capped absences    → mean:", round(df["absences_capped"].mean(), 2), "| std:", round(df["absences_capped"].std(), 2))

Original absences  → mean: 4.43 | std: 6.21
Capped absences    → mean: 4.29 | std: 5.28


<h3 style="color:#16A34A;border-left:6px solid #16A34A;padding-left:12px;margin-top:22px;">2.6 Filtering and aggregating</h3>

In [13]:
# Students who actually received a final grade (G3 > 0). Used later to test the effect of zero grades.
df_graded = df[~df["zero_grade"]].copy()
print(f"All enrolments        : {len(df):,}")
print(f"Zero final grade (G3=0): {df['zero_grade'].sum():,}  ({df['zero_grade'].mean():.1%})")
print(f"Graded enrolments     : {len(df_graded):,}")

# Aggregation: subject-level performance summary
subject_summary = df.groupby("subject", observed=True).agg(
    enrolments=("G3", "size"),
    mean_G1=("G1", "mean"),
    mean_G2=("G2", "mean"),
    mean_G3=("G3", "mean"),
    median_G3=("G3", "median"),
    pass_rate=("pass_fail", lambda s: (s == "Pass").mean()),
    zero_grades=("zero_grade", "sum"),
    avg_absences=("absences", "mean"),
)
subject_summary

All enrolments        : 1,044
Zero final grade (G3=0): 53  (5.1%)
Graded enrolments     : 991


,enrolments,mean_G1,mean_G2,mean_G3,median_G3,pass_rate,zero_grades,avg_absences
subject,,,,,,,,
Math,395,10.91,10.71,10.42,11.00,0.67,38,5.71
Portuguese,649,11.40,11.57,11.91,12.00,0.85,15,3.66


<h3 style="color:#16A34A;border-left:6px solid #16A34A;padding-left:12px;margin-top:22px;">2.7 Final cleaned dataset</h3>

In [14]:
print(f"Cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns "
      f"({df.shape[1] - raw.shape[1]} new columns created)")
df.to_csv("student_performance_cleaned.csv", index=False)     # saved for reproducibility
df.isna().sum().sum()

Cleaned dataset: 1,044 rows × 46 columns (12 new columns created)


np.int64(0)

✅ **Pre-processing summary:** no missing values or duplicates were found; columns were renamed and decoded; data types were corrected; **12 derived features** (including a capped copy of absences) were added; outliers were examined and handled with a capped copy; and a "graded-only" subset plus a subject-level aggregation were created. The cleaned file is exported as `student_performance_cleaned.csv`.

<a id="3"></a>
<h1 style="background:linear-gradient(90deg,#FF8A00,#F8615A);color:white;padding:16px 24px;border-radius:12px;margin:34px 0 12px 0;box-shadow:0 3px 8px rgba(0,0,0,.18);">3️⃣ Data Exploration</h1>
This part uses **numbers and tables only** (no charts): summary statistics, `groupby`, pivot tables, and correlations. The goal is to find the patterns in the data.

| Section | Level | Tools |
|---|---|---|
| 3.1 | **Univariate** – one variable at a time | `describe`, skewness, frequency tables |
| 3.2 | **Bivariate** – final grade vs one factor | `groupby`, pivot tables |
| 3.3 | **Multivariate** – several variables together | correlation matrix, multi-level `groupby`, pivot tables |
| 3.4 | Exploration summary | key observations |

<h2 style="color:#E06A00;border-bottom:3px solid #E06A00;padding-bottom:6px;margin-top:30px;">3.1 Univariate Exploration</h2>

<h3 style="color:#E0443E;border-left:6px solid #E0443E;padding-left:12px;margin-top:22px;">📌 Numerical variables</h3>

In [15]:
num_summary = df[["age", "absences", "G1", "G2", "G3", "avg_grade", "grade_change",
                  "total_alcohol", "parents_edu_avg"]].agg(
    ["count", "mean", "median", "std", "min", "max", "skew", "kurt"]).T
num_summary

,count,mean,median,std,min,max,skew,kurt
age,1044.00,16.73,17.00,1.24,15.00,22.00,0.43,0.04
absences,1044.00,4.43,2.00,6.21,0.00,75.00,3.74,26.60
G1,1044.00,11.21,11.00,2.98,0.00,19.00,0.08,-0.33
G2,1044.00,11.25,11.00,3.29,0.00,19.00,-0.50,1.34
G3,1044.00,11.34,11.00,3.86,0.00,20.00,-0.99,1.74
avg_grade,1044.00,11.27,11.33,3.22,1.33,19.33,-0.29,0.24
grade_change,1044.00,0.13,0.00,2.28,-12.00,11.00,-1.89,6.59
total_alcohol,1044.00,1.72,1.29,0.93,1.00,5.00,1.64,2.47
parents_edu_avg,1044.00,2.50,2.50,1.01,0.00,4.00,0.02,-1.16


**Reading the table.** All three grades are centred near 11 out of 20, but `G3` has the largest spread (std ≈ 3.9). `absences` has a mean (4.4) well above its median (2) and a very high skewness, so it is strongly right-skewed and driven by a few students with many absences. `grade_change` averages close to 0, so grades do not move much during the year *on average*.

<h3 style="color:#E0443E;border-left:6px solid #E0443E;padding-left:12px;margin-top:22px;">📌 Categorical variables (frequency tables)</h3>

In [16]:
freq_cols = ["subject", "school", "gender", "address", "wants_higher_edu", "internet_access",
             "pass_fail", "performance_band", "trend"]
freq = pd.concat(
    {c: df[c].astype(str).value_counts().rename("count").to_frame()
            .assign(share_pct=lambda t: (t["count"] / len(df) * 100).round(1))
     for c in freq_cols}
)
freq

count  share_pct
subject          Portuguese              649      62.20
                 Math                    395      37.80
school           Gabriel Pereira         772      73.90
                 Mousinho da Silveira    272      26.10
gender           Female                  591      56.60
                 Male                    453      43.40
address          Urban                   759      72.70
                 Rural                   285      27.30
wants_higher_edu Yes                     955      91.50
                 No                       89       8.50
internet_access  Yes                     827      79.20
                 No                      217      20.80
pass_fail        Pass                    814      78.00
                 Fail                    230      22.00
performance_band Average (10-13)         520      49.80
                 Fail (0-9)              230      22.00
                 Good (14-16)            224      21.50
                 Excellent (17-20)        70       6.70
trend            Stable                  706      67.60
                 Improved                231      22.10
                 Declined                107      10.20

**Reading the table.** Portuguese makes up 62% of enrolments, school *Gabriel Pereira* 74%, female students 57%, urban homes 73%, students wanting higher education 91% and students with internet 79%. About **78% of enrolments pass** (`G3 ≥ 10`). Several groups are small (for example, only 89 students do not want higher education), which matters when we compare groups later.

<h2 style="color:#E06A00;border-bottom:3px solid #E06A00;padding-bottom:6px;margin-top:30px;">3.2 Bivariate Exploration</h2>

Each table below shows the **number of students, mean and median final grade and pass rate (%)** for the groups of one factor.

In [17]:
def grade_table(col):
    return (df.groupby(col, observed=True)["G3"]
              .agg(students="count", mean_G3="mean", median_G3="median",
                   pass_rate_pct=lambda s: (s >= 10).mean() * 100)
              .round(2))

for col in ["subject", "gender", "school", "address", "internet_access", "wants_higher_edu"]:
    print(f"▶ Final grade by {col}")
    display(grade_table(col))

▶ Final grade by subject


,students,mean_G3,median_G3,pass_rate_pct
subject,,,,
Math,395,10.42,11.00,67.09
Portuguese,649,11.91,12.00,84.59


▶ Final grade by gender


,students,mean_G3,median_G3,pass_rate_pct
gender,,,,
Female,591,11.45,12.00,78.85
Male,453,11.20,11.00,76.82


▶ Final grade by school


...

,students,mean_G3,median_G3,pass_rate_pct
address,,,,
Rural,285,10.60,11.00,72.98
Urban,759,11.62,12.00,79.84


▶ Final grade by internet_access


,students,mean_G3,median_G3,pass_rate_pct
internet_access,,,,
No,217,10.53,11.00,73.27
Yes,827,11.55,12.00,79.20


▶ Final grade by wants_higher_edu


,students,mean_G3,median_G3,pass_rate_pct
wants_higher_edu,,,,
No,89,8.35,9.00,48.31
Yes,955,11.62,12.00,80.73


In [18]:
for col in ["study_time", "failures", "mother_edu", "mother_job", "weekday_alcohol", "absence_group"]:
    print(f"▶ Final grade by {col}")
    display(grade_table(col))

▶ Final grade by study_time


,students,mean_G3,median_G3,pass_rate_pct
study_time,,,,
1,317,10.58,11.00,72.56
2,503,11.34,11.00,77.93
3,162,12.49,13.00,85.80
4,62,12.27,12.00,85.48


▶ Final grade by failures


,students,mean_G3,median_G3,pass_rate_pct
failures,,,,
0,861,12.05,12.00,85.02
1,120,8.43,10.00,51.67
2,33,7.48,8.00,33.33
3,30,6.80,8.00,30.00


▶ Final grade by mother_edu


...

,students,mean_G3,median_G3,pass_rate_pct
mother_job,,,,
At Home,194,10.47,11.00,73.71
Health,82,12.68,13.00,84.15
Other,399,11.02,11.00,76.19
Services,239,11.66,12.00,79.92
Teacher,130,12.21,12.50,82.31


▶ Final grade by weekday_alcohol


,students,mean_G3,median_G3,pass_rate_pct
weekday_alcohol,,,,
1,727,11.70,12.00,80.61
2,196,10.56,11.00,71.94
3,69,10.90,10.00,76.81
4,26,9.27,11.00,61.54
5,26,10.38,11.00,69.23


▶ Final grade by absence_group


,students,mean_G3,median_G3,pass_rate_pct
absence_group,,,,
None,359,10.89,12.00,76.60
1-5,368,11.91,11.00,83.42
6-10,202,11.67,11.00,78.71
11-20,92,10.35,10.00,66.30
20+,23,10.43,10.00,52.17


<h3 style="color:#E0443E;border-left:6px solid #E0443E;padding-left:12px;margin-top:22px;">📌 Pivot tables: the same factor split by subject</h3>

In [19]:
print("Mean G3 – subject × gender")
display(df.pivot_table(index="subject", columns="gender", values="G3", aggfunc="mean", observed=True).round(2))
print("Mean G3 – subject × school")
display(df.pivot_table(index="subject", columns="school", values="G3", aggfunc="mean", observed=True).round(2))
print("Mean G3 – subject × wants higher education")
display(df.pivot_table(index="subject", columns="wants_higher_edu", values="G3", aggfunc="mean", observed=True).round(2))
print("Mean G3 – subject × paid classes")
display(df.pivot_table(index="subject", columns="paid_classes", values="G3", aggfunc="mean", observed=True).round(2))

Mean G3 – subject × gender


gender,Female,Male
subject,,
Math,9.97,10.91
Portuguese,12.25,11.41


Mean G3 – subject × school

school,Gabriel Pereira,Mousinho da Silveira
subject,,
Math,10.49,9.85
Portuguese,12.58,10.65


Mean G3 – subject × wants higher education


wants_higher_edu,No,Yes
subject,,
Math,6.80,10.61
Portuguese,8.80,12.28


Mean G3 – subject × paid classes


paid_classes,No,Yes
subject,,
Math,9.99,10.92
Portuguese,11.95,11.21


<h3 style="color:#E0443E;border-left:6px solid #E0443E;padding-left:12px;margin-top:22px;">📌 Investigating the zero grades</h3>

In [20]:
zeros = df[df["zero_grade"]]
zero_profile = pd.DataFrame({
    "Students with G3 = 0": [len(zeros), zeros["absences"].mean(), zeros["G1"].mean(), zeros["G2"].mean()],
    "Students with G3 > 0": [len(df_graded), df_graded["absences"].mean(), df_graded["G1"].mean(), df_graded["G2"].mean()],
}, index=["count", "mean absences", "mean G1", "mean G2"]).round(2)
display(zero_profile)

print("Zero-grade students with 0 absences:", (zeros["absences"] == 0).sum(), "of", len(zeros))
print("Correlation absences ↔ G3 (all students)       :", round(df["absences"].corr(df["G3"]), 3))
print("Correlation absences ↔ G3 (excluding G3 = 0)   :", round(df_graded["absences"].corr(df_graded["G3"]), 3))

print("\nMean G3 by absence group")
display(pd.DataFrame({
    "all students": df.groupby("absence_group", observed=True)["G3"].mean(),
    "excluding zero grades": df_graded.groupby("absence_group", observed=True)["G3"].mean(),
}).round(2))

,Students with G3 = 0,Students with G3 > 0
count,53.00,991.00
mean absences,0.00,4.67
mean G1,7.40,11.42
mean G2,4.42,11.61


Zero-grade students with 0 absences: 53 of 53
Correlation absences ↔ G3 (all students)       : -0.046
Correlation absences ↔ G3 (excluding G3 = 0)   : -0.218

Mean G3 by absence group


,all students,excluding zero grades
absence_group,,
None,10.89,12.77
1-5,11.91,11.91
6-10,11.67,11.67
11-20,10.35,10.35
20+,10.43,10.43


**Reading the zero-grade tables.** Every one of the **53** students with `G3 = 0` has **0 recorded absences**, and their earlier grades were already low (mean G1 ≈ 7.4, G2 ≈ 4.4). They probably left the course or skipped the final evaluation. Including them hides the real effect of absences: the correlation with `G3` is only −0.05 for everyone, but about −0.2 once they are excluded.

<h2 style="color:#E06A00;border-bottom:3px solid #E06A00;padding-bottom:6px;margin-top:30px;">3.3 Multivariate Exploration</h2>

<h3 style="color:#E0443E;border-left:6px solid #E0443E;padding-left:12px;margin-top:22px;">📌 Correlation analysis</h3>

In [21]:
num_cols = ["age", "mother_edu", "father_edu", "travel_time", "study_time", "failures", "family_relation",
            "free_time", "go_out", "weekday_alcohol", "weekend_alcohol", "health", "absences", "G1", "G2", "G3"]
corr = df[num_cols].corr()

print("Correlation of every numeric variable with the final grade (G3), strongest first:")
display(corr["G3"].drop("G3").sort_values(key=abs, ascending=False).round(3).to_frame("r with G3"))

pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
             .rename("r").reset_index().rename(columns={"level_0": "variable_1", "level_1": "variable_2"}))
pairs["abs_r"] = pairs["r"].abs()
print("Ten strongest relationships between any two variables:")
display(pairs.sort_values("abs_r", ascending=False).head(10).drop(columns="abs_r").round(3).reset_index(drop=True))

Correlation of every numeric variable with the final grade (G3), strongest first:


,r with G3
G2,0.91
G1,0.81
failures,-0.38
mother_edu,0.20
study_time,0.16
father_edu,0.16
weekday_alcohol,-0.13
age,-0.12
weekend_alcohol,-0.12
travel_time,-0.10


Ten strongest relationships between any two variables:


,variable_1,variable_2,r
0,G2,G3,0.91
1,G1,G2,0.86
2,G1,G3,0.81
3,mother_edu,father_edu,0.64
4,weekday_alcohol,weekend_alcohol,0.63
5,go_out,weekend_alcohol,0.40
6,failures,G3,-0.38
7,failures,G2,-0.38
8,failures,G1,-0.37
9,free_time,go_out,0.32


**Reading the correlations.** Apart from the grades themselves (`G2`, `G1`), **`failures` is the strongest negative correlate of `G3`** (≈ −0.38). Parents' education and study time are the strongest positive ones (≈ +0.16 to +0.20). Alcohol, going out and age are mildly negative. Some predictors overlap with each other: `mother_edu` and `father_edu` are strongly correlated (≈ 0.64), as are `weekday_alcohol` and `weekend_alcohol` (≈ 0.63).

<h3 style="color:#E0443E;border-left:6px solid #E0443E;padding-left:12px;margin-top:22px;">📌 Multi-factor pivot tables and group-bys</h3>

In [22]:
print("Mean G3 – study time × past failures   (student counts in the second table)")
display(df.pivot_table(index="study_time", columns="failures", values="G3", aggfunc="mean").round(1))
display(df.pivot_table(index="study_time", columns="failures", values="G3", aggfunc="count"))

print("Mean G3 – weekday alcohol × weekend alcohol")
display(df.pivot_table(index="weekday_alcohol", columns="weekend_alcohol", values="G3", aggfunc="mean").round(1))

print("Share of students whose grade declined / stayed / improved (G3 vs G1), % by subject")
display((pd.crosstab(df["subject"], df["trend"], normalize="index") * 100).round(1))

print("Average grade per period and subject")
display(df.groupby("subject", observed=True)[["G1", "G2", "G3"]].mean().round(2))

print("Subject × school × address")
display(df.groupby(["subject", "school", "address"], observed=True)["G3"]
          .agg(students="count", mean_G3="mean", pass_rate_pct=lambda s: (s >= 10).mean() * 100).round(2))

Mean G3 – study time × past failures   (student counts in the second table)


failures,0,1,2,3
study_time,,,,
1,11.60,8.10,4.80,7.10
2,11.90,8.70,8.20,6.40
3,13.00,8.30,10.00,NaN
4,12.40,9.70,NaN,NaN


failures,0,1,2,3
study_time,,,,
1,237.00,52.00,10.00,18.00
2,422.00,52.00,17.00,12.00
3,143.00,13.00,6.00,NaN
4,59.00,3.00,NaN,NaN


Mean G3 – weekday alcohol × weekend alcohol


weekend_alcohol,1,2,3,4,5
weekday_alcohol,,,,,
1,11.80,11.80,11.40,11.30,11.10
2,8.50,10.50,11.50,9.70,9.80
3,18.00,10.50,10.40,11.10,10.70
4,9.00,5.50,8.00,10.10,10.00
5,11.00,10.00,NaN,NaN,10.40


...

trend,Declined,Stable,Improved
subject,,,
Math,18.20,62.00,19.70
Portuguese,5.40,71.00,23.60


Average grade per period and subject


,G1,G2,G3
subject,,,
Math,10.91,10.71,10.42
Portuguese,11.40,11.57,11.91


Subject × school × address


students  mean_G3  pass_rate_pct
subject    school               address                                  
Math       Gabriel Pereira      Rural          63     9.76          63.49
                                Urban         286    10.65          68.53
           Mousinho da Silveira Rural          25     8.88          60.00
                                Urban          21    11.00          66.67
Portuguese Gabriel Pereira      Rural          78    12.14          89.74
                                Urban         345    12.68          93.04
           Mousinho da Silveira Rural         119    10.39          69.75
                                Urban         107    10.93          70.09

<h2 style="color:#E06A00;border-bottom:3px solid #E06A00;padding-bottom:6px;margin-top:30px;">3.4 Exploration Summary</h2>

The tables point to these patterns:

| # | Observation from the tables |
|---|---|
| 1 | `G1` and `G2` are by far the best predictors of `G3` (r ≈ 0.81 and 0.91) |
| 2 | Past failures: mean `G3` of 12.05 (no failure) falls to 8.43 (one failure) and 6.80 (three failures) |
| 3 | Math is weaker than Portuguese (mean 10.4 vs 11.9; pass rate 67% vs 85%), and Math averages *fall* during the year while Portuguese averages *rise* |
| 4 | Students who want higher education average 11.6 against 8.4 for those who do not |
| 5 | The gender gap goes in opposite directions in the two subjects |
| 6 | Study time helps up to 5–10 hours per week, with no extra gain above that |
| 7 | The 53 zero-grade students all have 0 absences, which distorts the absences result |
| 8 | Urban, internet-connected, Gabriel Pereira and higher-parent-education groups score about 1 point higher |

These are **observations** from descriptive tables; they show association, not cause and effect.